In [30]:
import pickle
import warnings
import json
warnings.filterwarnings("ignore")

import pandas as pd
import numpy  as np

from sklearn.model_selection    import train_test_split
from pathlib                    import Path
from IPython.display            import display, HTML
from methods.FeatureSelection import (
                                        remove_low_variance_features, 
                                        nan_values, 
                                        const_feature,
                                        fill_missing_values,
                                        get_feature_intervals,
                                        compare_samples
                                    )

In [31]:
np.set_printoptions(precision=4, suppress=True)
pd.set_option("display.precision", 4)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")
display(HTML("<style>.container { width:99% !important; }</style>"))
pd.options.display.float_format = '{:,.3f}'.format
pd.set_option('display.max_columns', None)
# ширина каждого столбца (в символах)
pd.set_option('display.max_colwidth', None)

In [32]:
path = f'prepared_data/dataset_for_feature_selection.pickle'

# загрузка данных
with open(path, 'rb') as f:
    samples = pickle.load(f)

# проверим, какие ключи есть в загруженном словаре
print("Ключи в samples:", samples.keys())

Ключи в samples: dict_keys(['df', 'info_fields', 'features', 'cat_features', 'num_features', 'target'])


In [33]:
df           = samples['df']
info         = samples['info_fields']
cat_features = samples['cat_features']
num_features = samples['num_features']
target       = samples['target']

In [34]:
df.shape

(144900, 37)

## Анализ вещественных фичей

In [35]:
feature_reasons = {}
low_var_features                         = remove_low_variance_features(df, num_features, threshold=0.001)
feature_reasons['low_variance_features'] = low_var_features
num_features                             = list(set(num_features) - set(low_var_features))

open                            310,789,038.342
high                            317,795,669.792
low                             303,018,378.224
close                           310,751,148.492
volume           12,569,952,105,276,729,344.000
ret_1                                     0.000
ret_3                                     0.000
ret_5                                     0.002
ret_20                                    0.010
vol3                                      0.000
vol_5                                     0.000
vol_15                                    0.000
vol_30                                    0.000
vol_45                                    0.000
vol_90                                    0.000
mom_3                                     0.001
mom_5                                     0.002
mom_15                                    0.007
mom_20                                    0.010
sma_3                           310,861,493.277
sma_5                           311,226,

In [36]:
# удаление константных фичей
const_features                    = const_feature(df, num_features,  0.9)
num_features                      = list(set(num_features) - set(const_features))
feature_reasons['const_features'] = const_features

In [37]:
# удаление фичей с пропусками
nan_features                    = nan_values(df, num_features, 0.2)
num_features                    = list(set(num_features) - set(nan_features))
feature_reasons['nan_features'] = nan_features

# кол-во пропусков по данным фичам
if nan_features:
    (df[nan_features].isna().sum() / df.shape[0]).sort_values(ascending=False)

In [38]:
reasons = []
for reason, features in feature_reasons.items():
    for feature, explanation in features.items():
        reasons.append({
            "reason"     : reason,
            "feature"    : feature,
            "explanation": explanation
        })

reasons_df = pd.DataFrame(reasons)

In [39]:
reasons_df

,reason,feature,explanation
0,low_variance_features,ret_1,0.000
1,low_variance_features,ret_3,0.000
2,low_variance_features,vol3,0.000
3,low_variance_features,vol_5,0.000
4,low_variance_features,vol_15,0.000
5,low_variance_features,vol_30,0.000
6,low_variance_features,vol_45,0.000
7,low_variance_features,vol_90,0.000
8,low_variance_features,hl_range,0.000
9,const_features,is_month_end,"Мажоритарное значение 0, доля значения - 0.97"


In [40]:
# очищаем лишние поля
features = num_features + cat_features
# df = df[info + features + [target]]

# Удаляем записи, в которых остались пропуски

In [41]:
df.isna().sum()

date                 0
open                 0
high                 0
low                  0
close                0
volume               0
ticker               0
ret_1               69
ret_3               69
ret_5              345
ret_20            1380
vol3               207
vol_5              345
vol_15            1035
vol_30            2070
vol_45            3105
vol_90            6192
mom_3              207
mom_5              345
mom_15            1035
mom_20            1380
sma_3              138
sma_5              276
sma_10             621
sma_20            1311
sma_ratio         1311
vol_mean_5         276
vol_mean_15        966
vol_mean_45       3036
vol_ratio          276
hl_range             0
dow                  0
month                0
week                 0
is_month_end         0
is_month_start       0
target               0
dtype: int64

In [42]:
df = df.dropna()

In [43]:
df.isna().sum()

date              0
open              0
high              0
low               0
close             0
volume            0
ticker            0
ret_1             0
ret_3             0
ret_5             0
ret_20            0
vol3              0
vol_5             0
vol_15            0
vol_30            0
vol_45            0
vol_90            0
mom_3             0
mom_5             0
mom_15            0
mom_20            0
sma_3             0
sma_5             0
sma_10            0
sma_20            0
sma_ratio         0
vol_mean_5        0
vol_mean_15       0
vol_mean_45       0
vol_ratio         0
hl_range          0
dow               0
month             0
week              0
is_month_end      0
is_month_start    0
target            0
dtype: int64

In [44]:
# приводим все категориальные фичи к строковому типу
df[cat_features] = df[cat_features].astype('str')

## Разбиваем на подвыборки

In [45]:
# очищаем лишние поля которые могли оставться в датафрейме послое фичи селекшена
merged_list = info + num_features + cat_features + [target]
result_df   = df[merged_list].copy()
print('Размер: ', result_df.shape)
result_df.head(2)

Размер:  (138708, 26)


,date,ticker,week,vol_mean_45,mom_20,sma_3,sma_20,vol_mean_5,open,volume,mom_5,close,month,low,vol_mean_15,dow,ret_20,vol_ratio,mom_15,sma_ratio,sma_10,sma_5,ret_5,mom_3,high,target
90,2014-10-15,AFKS,42.000,"20,261,673.333",-0.282,14.167,16.909,"19,560,300.000",15.090,"12,481,000.000",-0.095,13.850,10,13.740,"30,501,106.667",2.000,-0.282,0.638,-0.250,-0.139,14.715,14.080,-0.095,0.041,15.260,0
91,2014-10-16,AFKS,42.000,"20,414,342.222",-0.282,14.050,16.482,"16,218,400.000",14.110,"13,589,000.000",-0.089,13.300,10,13.100,"29,109,580.000",3.000,-0.282,0.838,-0.250,-0.139,14.635,13.820,-0.089,-0.026,14.140,1


In [46]:
build, test  = train_test_split(result_df, stratify=result_df[target], test_size=0.2, random_state=42)
train, valid = train_test_split(build, stratify=build[target], test_size=0.2, random_state=42)

## Сравнение выборок на схожесть фичей

In [47]:
# колонки для сравнения compare
features_to_compare = num_features + cat_features

In [48]:
train_valid_comparison_df = compare_samples(train, valid, features_to_compare).reset_index().rename({'index': 'feature'}, axis = 1)[['feature', 'euclidean']].sort_values(by = 'euclidean', ascending = False)
build_test_comparison_df  = compare_samples(build, test, features_to_compare).reset_index().rename({'index': 'feature'}, axis = 1)[['feature', 'euclidean']].sort_values(by = 'euclidean', ascending = False)

In [49]:
train_valid_comparison_df

,feature,euclidean
13,dow,100.000
15,vol_ratio,99.569
17,sma_ratio,99.545
16,mom_15,99.280
20,ret_5,99.248
8,mom_5,99.248
2,mom_20,99.177
14,ret_20,99.177
21,mom_3,99.155
0,week,98.837


In [50]:
build_test_comparison_df

,feature,euclidean
10,month,100.000
17,sma_ratio,99.421
15,vol_ratio,99.371
0,week,99.252
14,ret_20,99.201
2,mom_20,99.201
16,mom_15,99.136
21,mom_3,99.042
11,low,98.853
9,close,98.784


In [51]:
drop_features = []

train = train.drop(drop_features, axis = 1, errors = 'ignore')
valid = valid.drop(drop_features, axis = 1, errors = 'ignore')
test  = test.drop(drop_features, axis = 1, errors = 'ignore')

cat_features = [
    col for col in train.drop(info + [target], axis = 1).columns.to_list()
    if col in train.columns and train[col].dtype == "object"
]

In [52]:
train.shape,valid.shape, test.shape, build.shape

((88772, 26), (22194, 26), (27742, 26), (110966, 26))

In [53]:
#словарь с ключами:
#      cat_features
#  - unique_list уникальные значение на build выборки
#  - freq_value частовстречаемое значение на build выборки
#  - default_value значение, на которое нужно заменить категорию, которая не вошла в unique_list
# 
#      num_features
#  - min
#  - max
#  - mean



features_intervals = get_feature_intervals(build[num_features + cat_features])
features_intervals

{'week': {'min': np.float64(1.0),
  'max': np.float64(52.0),
  'mean': np.float64(27.045)},
 'vol_mean_45': {'min': np.float64(1691.517),
  'max': np.float64(35195291413.333),
  'mean': np.float64(771473134.984)},
 'mom_20': {'min': np.float64(-0.282),
  'max': np.float64(0.334),
  'mean': np.float64(0.008)},
 'sma_3': {'min': np.float64(0.008),
  'max': np.float64(144745.933),
  'mean': np.float64(3276.411)},
 'sma_20': {'min': np.float64(0.008),
  'max': np.float64(144889.3),
  'mean': np.float64(3283.317)},
 'vol_mean_5': {'min': np.float64(1253.846),
  'max': np.float64(30254681299.999),
  'mean': np.float64(673193344.424)},
 'open': {'min': np.float64(0.008),
  'max': np.float64(144800.4),
  'mean': np.float64(3277.02)},
 'volume': {'min': np.float64(800.0),
  'max': np.float64(29501215900.0),
  'mean': np.float64(642906531.193)},
 'mom_5': {'min': np.float64(-0.139),
  'max': np.float64(0.155),
  'mean': np.float64(0.002)},
 'close': {'min': np.float64(0.008),
  'max': np.float64

In [54]:
# пришла пустота на инференс ты меняешь на наиболее вероятное/популярное (для категориальной)  для числовой на среднее
# если числовая пришла выше границы, то замечанием на верзнюю границу если нижнняя на нижнюю

# если пришло значение но его нет в словаре заменяем на other
# как быть если не обучалась на other но пришло на инфуренс категориальное значение которго не было на обучении
# заменять на наиболее вероятное

#  замена новых значений в test 
for feature in features_intervals.keys():
    if feature in cat_features:
        test.loc[test[feature].isna(), feature] = features_intervals[feature]['freq_value']
        test.loc[~ test[feature].isin(features_intervals[feature]['unique_list']), feature] = features_intervals[feature]['default_value']
    else:
        test.loc[test[feature].isna(), feature] = features_intervals[feature]['mean']
        test.loc[test[feature]>features_intervals[feature]['max'], feature] = features_intervals[feature]['max']
        test.loc[test[feature]<features_intervals[feature]['min'], feature] = features_intervals[feature]['min']

In [55]:
path = f'prepared_data/data_for_modeling.pickle'

samples = {
    'train'       : train,
    'valid'       : valid,
    'test'        : test,
    'target'      : target,
    'reasons_df'  : reasons_df,
    'info_fields' : info,
    'features'    : train.drop(info + [target], axis = 1).columns.to_list(),
    'cat_features': cat_features,
    'features_intervals' : features_intervals
}
with open(path, 'wb') as f:
    pickle.dump(samples, f)